# Threshold Crossing Sample Count

[image-6.png](attachment:image-6.png)
[image-4.png](attachment:image-4.png)

### ✅ ❌ ⏳ 📋 🟡



> Priorities 

- ✅ Minimal QRX files
- ✅ QRX in Parquet format !!
- ❌ NA handling, check if PRQ is indentical with TSV
- ⏳ What is Point in buffers ?
- ✅ Why is bfx writing so slow -> moved to parquet
- ✅ Stopwatch for loading times of record, speed it up  
- ✅ Add FFT plot to vta.py from VFIB.ipynb
- ✅ Create Processor class to group Detectors
- 📋 CFM as tuple in Process result
- ✅ MERGE implementation of Gusev / SPEC algo with TCSC
- 🟡 BRURE FORCE param search, optimal ROC
- 🟡 Write results to files with Detector config ...

> Hilbert Transform

- 📋 Use instanteous phase and frequency to determine if the signal is oscillatory or not.  
- 📋 How to implement this with FIR filter, what np.angle does? Why is ifreq with negative spikes?
  
> analytic = signal.hilbert(sig)  
> dt = 1 / FS  
> angle = np.angle(analytic) # type: ignore # gives you the instantaneous phase (in radians).  
> ifreq = np.diff(angle) / (2 * np.pi * dt) # gives you the instantaneous frequency (in Hz).  


In [ ]:
# IMPORTS ###########################################

import importlib
import warnings
warnings.filterwarnings("ignore")

#####################################################

import numpy as np
from scipy import signal

from pxg import plot
importlib.reload(plot)

import vta
importlib.reload(vta)
from vta import Detector, EVAL
from vta.dsp import LynnFilter

from pxg import Stop
from pxg import FS, MS
from pxg import EXG, Record

%config InlineBackend.figure_format = "retina"

The ECG signal is preprocessed using the same well-known filtering process as used in [Reliability of old and new ventricular fibrillation detection
algorithms for automated external defibrillators](http://www.biomedical-engineering-online.com/content/4/1/60).

The filtering algorithm works in four successive steps.

- First, the mean value is subtracted from the signal.
- Second, a moving average filter of order 5 is applied in order to remove high frequency noise like interspersions and muscle noise.
- Third, a drift suppression is carried out by a high pass filter with a cut-off frequency of 1 Hz.
- In the last step, a low-pass Butterworth filter with a cut-off frequency of 30 Hz is applied which suppresses the high frequency information even more.

In [ ]:
def PaperFilter(y: np.ndarray) -> np.ndarray:
    ## Remove mean ####################
    y = y - np.mean(y)
    ## Order five moving average ######
    y = np.convolve(y, np.ones(5)/5, mode="same")
    ###################################

    ## High pass filter ###############
    b, a = signal.butter(2, 1 / (FS / 2), btype='high') # type: ignore
    y = signal.filtfilt(b, a, y) # type: ignore
    ###################################

    ## Butter low pass 30 Hz ##########
    b, a = signal.butter(4, 30 / (FS / 2), btype='low') # type: ignore
    y = signal.filtfilt(b, a, y) # type: ignore
    ###################################
    return np.array(y)
pass #def

In [ ]:
### TCSC ###

CU_SEN = 79.74    
CU_SPC = 88.14
CU_PPV = 65.02
CU_ACC = 86.32
CU_F1V = CU_SEN * CU_PPV * 2 / (CU_SEN + CU_PPV)

MT_SEN = 97.48
MT_SPC = 99.33
MT_PPV = 18.98
MT_ACC = 99.33
MT_F1V = MT_SEN * MT_PPV * 2 / (MT_SEN + MT_PPV)

class TCSC(Detector):
    def __init__(self, SEG = 3, WIN = 8, INC = 1, AMP = 20, THR = 48, EPI = 50):
        super().__init__(SEG = SEG, WIN = WIN)
        self.INC = INC
        self.AMP = AMP
        self.THR = THR
        self.EPI = EPI

        self.COS = signal.windows.tukey(self.SEGFS, alpha=(0.5 / self.SEG)) # type: ignore

        self.Refs["cudb"]  = (f"REF\t***\t***\t***\t***\t{CU_ACC:.2f}\t{CU_SPC:.2f}\t***\t{CU_SEN:.2f}\t{CU_PPV:.2f}\t{CU_F1V:.2f}")
        self.Refs["mitdb"] = (f"REF\t***\t***\t***\t***\t{MT_ACC:.2f}\t{MT_SPC:.2f}\t***\t{MT_SEN:.2f}\t{MT_PPV:.2f}\t{MT_F1V:.2f}")
    pass #def

    def Clone(self, SEG = None, WIN = None, INC = None, AMP = None, THR = None, EPI = None):
        SEG = self.SEG if SEG is None else SEG
        WIN = self.WIN if WIN is None else WIN
        INC = self.INC if INC is None else INC
        AMP = self.AMP if AMP is None else AMP
        THR = self.THR if THR is None else THR
        EPI = self.EPI if EPI is None else EPI
        return TCSC(SEG, WIN, INC, AMP, THR, EPI)
    pass #def

    # def count(self, seg: np.ndarray) -> int:
    #     seg = seg * self.COS
    #     val = np.max(np.abs(seg)) * self.AMP / 100
    #     return np.sum(np.sign(np.maximum(0, np.abs(seg) - val)))
    # pass #def

    def Eval(self, sig: np.ndarray, off = 0) -> tuple[np.ndarray, np.ndarray]:
        # Calculate the moving maximum using a maximum filter
        # max_filt = ndimage.maximum_filter(sig, size=W, mode='nearest')
        # return max_filt
        off = off * FS
        end = off + self.SEGFS
        thr = np.zeros(len(sig))
        dig = np.zeros(len(sig))
        while end < len(sig):
            seg = sig[off:end] * self.COS
            # man = seg - np.mean(seg)
            asg = np.abs(seg)
            val = np.max(asg) * self.AMP / 100
            thr[off:end] = val
            dig[off:end] = np.sign(np.maximum(0, asg - val))
            off += self.SEGFS
            end += self.SEGFS
        return thr, dig
    pass #def

    def Detect(self, rec: Record, ref: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        rec.Basic = LynnFilter(rec.Point, W = 12)

        m0, d0 = self.Eval(rec.Local, 0)
        m1, d1 = self.Eval(rec.Local, 1)
        m2, d2 = self.Eval(rec.Local, 2)

        m1[0:FS] = m0[0:FS]
        m2[0:2*FS] = m1[0:2*FS]

        rec.Digit = d0
        rec.Maxim = (m0 + m1 + m2) / 3

        secs = len(rec.Local) // FS
        decs = secs - self.WIN + 1
        ress = np.zeros(decs)
        claz = np.zeros(len(rec.Local))

        for d in range(0, decs, self.INC):
            sum = 0
            rum = 0
            cnt = 0
            for s in range(self.STEPS):
                ix = d + s
                off = ix * FS
                end = off + self.SEGFS
                if ix % 3 == 0:
                    g = d0
                elif ix % 3 == 1:
                    g = d1
                else:
                    g = d2
                pass #if
                sum += np.sum(g[off:end])
                # rum += self.count(rec.Local[off:end]) / self.SEGFS
                cnt += self.SEGFS
            pass #for
            
            off = d * FS
            end = off + self.WINFS

            par = sum / cnt

            # if abs(par - rum / self.STEPS) > 0.00001:
            #     raise ValueError(f"Sum and Rum differ at d={d}: {par} vs {rum / self.STEPS}")
            # pass #if

            val = 0
            if par > self.THR / 100:
                val = 1
                claz[off:end] = 1
            pass #if

            rcn = np.sum(ref[off:end])
            if rcn > self.WINFS * self.EPI / 100:
                # print(rcn, self.WINFS / 2)
                rcn = 1
            elif rcn != 0:
                # print(rcn, self.WINFS / 2)
                rcn = 0
            pass #if
                
            ress[d] = rcn * 2 + val
        pass #for

        rec.Local = np.abs(rec.Local)
    
        return claz, ress
    pass #def
pass #class

In [ ]:
vta.ALIGN = False
vta.REPORTW = True
vta.FILTER = PaperFilter

# EXG('ahadb', learn=True, cfm=True, bfx = True, var_qrx=True, prime = True)
# EXG('mitdb', learn=True, cfm=True, bfx = True, var_qrx=True, prime = True)
# EXG('cudb',  learn=True, cfm=True, bfx = True, var_qrx=True, prime = True)
# EXG('vfdb',  learn=True, cfm=True, bfx = True, var_qrx=True, prime = True)

pass

In [ ]:
EVAL(TCSC(), "mitdb", chart=True)
pass

In [ ]:
EVAL(TCSC(), "cudb", chart=False)
pass

In [ ]:
EVAL(TCSC(), "ahadb", chart=False)
pass

In [ ]:
EVAL(TCSC(), "vfdb", chart=False)
pass